In [ ]:
# Ensure jacopy is importable when this notebook is opened
# directly (not via pytest). Walks up from the notebook's
# directory to the repo root and prepends it to sys.path if
# jacopy isn't already installed into this kernel.
try:
    import jacopy  # noqa: F401
except ModuleNotFoundError:
    import sys
    from pathlib import Path
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "jacopy" / "__init__.py").is_file():
            sys.path.insert(0, str(candidate))
            break
    import jacopy  # noqa: F401

# 24, Writing your own Problem wrapper

Companion notebook to [24_custom_problem_wrapper.md](24_custom_problem_wrapper.md). Five-step recipe walked end-to-end via a worked `AlmostSymplecticProblem` (non-degenerate but **not** closed `ω`), illustrates every moving part without overlapping any existing wrapper.

## The five steps

1. Pick the geometric data the wrapper carries.
2. Auto-declare structural axioms on the registry (check `has(...)` first, pre-declared flags take precedence).
3. Assemble the engine, layer your rules onto `default_engine(registry=…)`.
4. Write builder + prover methods.
5. (optional) Register seeded theorems for one-step citations.

## Worked example, `AlmostSymplecticProblem(ω)`

An almost-symplectic form is non-degenerate but not necessarily closed. We get the vector-field-equality rule (`ι_X ω = ι_Y ω ⇒ X = Y`) but *not* the Hamiltonian / Poisson structure that closure unlocks.

In [ ]:
from typing import Optional
from jacopy.algebra.derivation import Act
from jacopy.calculus.interior import interior
from jacopy.calculus.nondegenerate_axioms import (
    NonDegenerateInteriorEqualityDefinition,
)
from jacopy.core.expr import Expr
from jacopy.core.properties import Graded, NonDegenerate
from jacopy.core.registry import PropertyRegistry
from jacopy.proof.expansion import ExpansionEngine, default_engine

class AlmostSymplecticProblem:
    """`(ω, registry)`, non-degenerate (not necessarily closed) 2-form."""

    __slots__ = ("_omega", "_registry", "_engine", "_name")

    def __init__(self, omega, *, registry=None, name=None):
        self._omega    = omega
        self._registry = registry or PropertyRegistry()
        # Step 2, auto-declare; pre-declared flags take precedence.
        if not self._registry.has(omega, Graded):
            self._registry.declare(omega, Graded(degree=2))
        if not self._registry.has(omega, NonDegenerate):
            self._registry.declare(omega, NonDegenerate())
        # Step 3, engine assembly.
        base = default_engine(registry=self._registry)
        self._engine = ExpansionEngine(
            list(base.definitions) + [
                NonDegenerateInteriorEqualityDefinition(
                    registry=self._registry,
                ),
            ]
        )
        self._name = name or f"AlmostSymplecticProblem({omega._repr_inner()})"

    @property
    def omega(self): return self._omega
    @property
    def registry(self): return self._registry
    @property
    def engine(self): return self._engine
    @property
    def name(self): return self._name

    # Step 4, builders + provers.
    def musical_flat(self, X):
        """`ω^♭(X) = ι_X ω`."""
        return Act(interior(X), self._omega)


## Try it, the rule fires on the difference

On the obstruction `ι_X ω − ι_Y ω`, the engine reduces to `X − Y` in one step. That **is** the proof transcript of `ι_X ω = ι_Y ω ⇒ X = Y`, the residue `X − Y` is the equality the caller discharges.

In [ ]:
from jacopy.core.expr import Symbol, Sum, Neg

omega = Symbol("ω")
X = Symbol("X"); Y = Symbol("Y")
reg = PropertyRegistry()
reg.declare(X, Graded(degree=1))
reg.declare(Y, Graded(degree=1))

prob = AlmostSymplecticProblem(omega, registry=reg)
print(f"name         : {prob.name}")
print(f"engine rules : {len(prob.engine.definitions)}")

obstruction = Sum(prob.musical_flat(X), Neg(prob.musical_flat(Y)))
out, steps = prob.engine.expand(obstruction)
print(f"input  : {obstruction}")
print(f"output : {out}")
print(f"rule   : {steps[0].rule}")

## Picking your axioms, flags vs Definitions

| Mechanism | Use when | Example |
|---|---|---|
| Registry flag | Property is a one-bit fact about a single object | `Closed`, `NonDegenerate`, `Poisson` |
| Custom `Definition` subclass | Property is a non-trivial rewrite shape | tilde closure axioms |
| Seeded `Theorem` | Identity should appear as a single citation step | `poisson_jacobi`, `courant_dorfman_bridge` |

Prefer flags when you can, declarative, opt-out via pre-declaration, single rule fires per object.

## Where to look in the codebase

| Source | Read for |
|---|---|
| `library/symplectic.py` | Smallest non-trivial wrapper |
| `library/courant_algebroid.py` | Wrapper with seeded theorems + bridge identity |
| `library/bianchi_problem.py` | Custom proof loop (`_expand_to_canonical`) |
| `library/koszul_problem.py` | Largest wrapper, multi-engine + canonicalize_indices pre-pass |
| `library/cartan_structure.py` | Index-laden wrapper with per-problem registry |

Read the wrapper closest to your shape; the pattern repeats.

## Summary

* Five-step recipe: data → axioms → engine → methods → (seeded theorems).
* `registry.has(...)` check is the **opt-out mechanism**: pre-declared flags take precedence.
* `default_engine(registry=…)` is the right base for most form-side problems; layer your rules onto its `definitions` list.
* Engine order: definitions before linearity; frame-scoped rules before generic; opt-in rules for loop-prone pairs (decomp + duality).
* Read `library/symplectic.py` for the smallest template, `library/koszul_problem.py` for the largest.